# Phase 5 — Baseline LM (Step 1 curriculum)

**Requires:** Phase 2 data at `data/wikitext103/processed/sequences_*.jsonl`

## Right-panel settings (mandatory)
1. **Accelerator → GPU T4 x2 or P100**
2. **Persistence → Files Only**
3. Internet **On** (first time only, to download `gpt2`)

Full run ≈ **1–3 hours** on T4 for 3000 steps.  
Smoke test ≈ **5–10 minutes** (50 steps).

After training: **Save Version → Save & Run All (Commit)** or you lose the model.

In [ ]:
import os, sys
REPO = 'https://github.com/Yash-0525/sentiment_bias_project.git'
BRANCH = 'arena/01a0a078-sentiment-bias-project'
DEST = '/kaggle/working/sentiment_bias_project'
if os.path.isdir(DEST):
    !cd {DEST} && git fetch --quiet origin && git checkout {BRANCH} && git pull --quiet origin {BRANCH}
else:
    !git clone --branch {BRANCH} --single-branch --quiet {REPO} {DEST}
sys.path.insert(0, DEST)
%cd {DEST}
!git log -1 --oneline
from src import paths
paths.bootstrap(verbose=True)
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
!python tests/test_phase5_unit.py

In [ ]:
from pathlib import Path
p = Path('data/wikitext103/processed/sequences_train.jsonl')
print('train data exists:', p.is_file(), p)
if p.is_file():
    print('size MB', round(p.stat().st_size/1024**2, 1))
assert p.is_file(), 'Phase 2 data missing — restore from previous Commit or re-run Phase 2'

## SMOKE TEST (run this FIRST — ~5–10 min)

Confirms train loop, checkpoint, PPL. Then run FULL.

In [ ]:
!python -m src.train_baseline \
  --config configs/baseline.yaml \
  --run-name baseline_smoke \
  --max-steps 50 \
  --eval-every 25 \
  --save-every 25 \
  --max-train-sequences 1024 \
  --max-val-sequences 128 \
  --micro-batch-size 2 \
  --grad-accum-steps 4 \
  --no-resume

In [ ]:
from pathlib import Path
import json
root = Path('models/baseline_smoke')
print('tags:', sorted([p.name for p in root.iterdir() if p.is_dir()]) if root.is_dir() else None)
best = root / 'best' / 'state.json'
latest = root / 'latest' / 'state.json'
for p in (best, latest):
    if p.is_file():
        print(p, json.loads(p.read_text()))
assert (root/'best'/'hf_model').is_dir() or (root/'latest'/'hf_model').is_dir()
print('SMOKE CHECKPOINT OK')

## FULL BASELINE TRAIN (~1–3 h)

Only after smoke works. Leave the tab open or use **Save & Run All**.
Resume is ON — if the session dies, re-run this cell.

In [ ]:
!python -m src.train_baseline \
  --config configs/baseline.yaml \
  --run-name baseline

In [ ]:
from pathlib import Path
import json

summary = Path('results/baseline/phase5_summary.json')
assert summary.is_file(), summary
s = json.loads(summary.read_text())
print(json.dumps(s, indent=2))

best_model = Path('models/baseline/best/hf_model')
latest_model = Path('models/baseline/latest/hf_model')
assert best_model.is_dir() or latest_model.is_dir()

state = Path('models/baseline/best/state.json')
if state.is_file():
    st = json.loads(state.read_text())
    print('best step', st.get('step'), 'best_val_ppl', st.get('best_val_ppl'))
    assert st.get('step', 0) >= 500 or s.get('final_step', 0) >= 50

print('PHASE 5 VERIFICATION: PASS')
print('NEXT: Save Version → Save & Run All (Commit) before closing')